# GMM binder / non-binder hypothesis test — HADDOCK & Rosetta energies vs the biological GA-importer labels

**Kernel:** `abcfold-npf-notebook` (`envs/notebook.yaml`)

## The hypothesis (from the PI discussion)

> The published biological substrate data for gibberellin (GA) transport in the
> *Arabidopsis* NPF family is noisy and poorly reproducible. Some proteins
> currently annotated as **non-importers** may in fact be real GA transporters
> ("contamination" of the negative class), and the true positives are almost
> certainly **graded, not binary** — NPFs in the same clade differ by a lot in
> how fast they move GA.

**What this notebook does.** For the 143 HADDOCK3-redocked GA1 complexes
(`redocking/`) that already have a PyRosetta REF2015 rescoring
(`rescoring/src/rescore_redocked_batch.py`), fit **unsupervised Gaussian
mixture models** to the energy observables — HADDOCK physical score, Rosetta
whole-pose `total_score`, Rosetta ligand↔pocket two-body sum — with the
biological labels **hidden**. Then ask:

1. Does any single energy axis fall into a clean **bimodal** (binder /
   non-binder) distribution at all?
2. If it does, does the unsupervised split **line up** with the biological
   GA-importer call — here the **strict high-confidence split**
   (`NPF_LDA_kernel/config/config.yaml` `hc_importers` / `hc_non_importers`,
   29 of the 48 redocked proteins)?
3. Which proteins does the energy call **differently** from the HC label — i.e.
   the concrete candidate-contamination list to take back to the PI.
4. Is a 2-D (HADDOCK × Rosetta) mixture any cleaner than 1-D?
5. How much of the spread is **clade** structure (the "same clade, very
   different transport rate" confounder)?
6. Supervised upper bound: best separation any of these energies can achieve
   against the labels, next to the sequence-only LDA kernel (near-perfect on
   this split — `hc/summary.tsv` LOO AUC 0.996–1.00).

## Caveats baked in before we start — read these

- **HADDOCK's restraints are identical for importer and non-importer.**
  `redocking/src/define_active_passive.py` builds the ambiguous interaction
  restraints (AIRs) from the *same* 35 CDD pocket residues for every protein,
  so HADDOCK is *driven* toward the same pocket regardless of true importer
  status. `redocking/RESULTS.md` §4 shows importer vs non-importer barely
  differ on pocket-engagement metrics for exactly this reason.
- **Off-target poses are filtered out first (§1a).** A bimodal HADDOCK-score
  split is otherwise **more likely to be pose quality** (on-target vs
  membrane/off-target pose) than biology; earlier runs confirmed that. This
  version drops every complex whose scored pose has `< 3` CDD-pocket contacts
  before any GMM is fit, so the pose-quality explanation is removed by
  construction rather than only tested for.
- **The two force fields disagree on sign.** `redocking/RESULTS.md` §4:
  HADDOCK rates importer poses *more* favourable, an independent PyRosetta
  REF2015 rescoring of the same poses rates them *less* favourable. Neither
  should be trusted alone.
- **The labels are the thing under suspicion.** "Agreement with the label" is
  therefore not the only success criterion — a clean bimodal split that
  *disagrees* with a handful of labels in a chemically sensible direction is
  exactly what the hypothesis predicts.
- **`role` in the redocking manifest is not independent of the label** — it
  was derived from overlapping curated lists. We use the strict
  `hc_importers` / `hc_non_importers` split as the canonical biological label
  (`bio_label`), keep the original `labels.tsv` call as `assay_label` for
  cross-reference, and treat `role` as a convenience alias.
- **The strict split is small and clade-skewed.** 29 redocked proteins (10
  importer / 19 non-importer): the importers are NPF1×2, NPF2×7, NPF3×1; the
  non-importers are entirely NPF5–NPF8. "Importer vs non-importer" here is
  almost exactly "NPF1/2/3 vs NPF5/6/7/8" — a near-total clade confound, worse
  than on the full label set — so read §7 (clade) alongside every
  label-agreement number.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import adjusted_rand_score, roc_auc_score, balanced_accuracy_score
from scipy import stats

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 120)

ROOT = Path("..").resolve()               # repo root (.resolve() so .parent works)
REDOCKING = ROOT / "redocking"
PER_COMPLEX = REDOCKING / "results" / "rescoring" / "per_complex"
COMPARISON = REDOCKING / "results" / "comparison"
LDA_KERNEL = ROOT.parent / "NPF_LDA_kernel" / "results" / "ga_classifier"   # sibling project

RNG = 0
np.random.seed(RNG)

# Energy metrics we test. For all of them LOWER = more binder-like, so the
# favourable GMM component is always the low-mean one and the ROC score is -x.
METRICS = {
    "haddock_score":            "HADDOCK3 physical score of the top model (vdW+elec+desolv+AIR), REU-eq",
    "total_score_haddock_pose": "PyRosetta REF2015 whole-pose total_score of the redocked pose, REU",
    "lig_twobody_sum":          "sum of Rosetta ligand<->pocket-residue two-body totals, REU",
    "lig_twobody_min":          "single most favourable Rosetta ligand<->residue two-body total, REU",
    "fa_rep_haddock_pose":      "Rosetta fa_rep of the redocked pose (steric-clash proxy), REU (lower = cleaner)",
}
ROLE_COLORS = {"importer": "#2ca02c", "non_importer": "#7f7f7f", "ambiguous": "#d62728"}

## 1. Assemble the per-complex energy table

One row per redocked complex = per `(protein, ca_cluster)` macro-conformation.
`rescore_redocked_batch.py` scores exactly **one** pose per complex (HADDOCK3's
own top-ranked model, no FastRelax, no replicas), so every number here is that
single real pose's energy — no ensemble averaging.

In [ ]:
def load_complex(fp: Path) -> dict:
    d = pd.read_csv(fp)
    head = d.iloc[0]
    per_res = d.drop_duplicates("prot_resi")
    return {
        "complex_id": head.complex_id,
        "protein": head.protein,
        "role": head.role,
        "form": head.form,
        "ca_cluster": int(head.ca_cluster),
        "haddock_score": float(head.haddock_score),
        "total_score_haddock_pose": float(head.total_score_haddock_pose),
        "fa_rep_haddock_pose": float(head.fa_rep_haddock_pose),
        "lig_twobody_sum": float(per_res["twobody_total"].sum()),
        "lig_twobody_min": float(per_res["twobody_total"].min()),
        "n_contact_res": int(per_res["prot_resi"].nunique()),
    }


complex_df = pd.DataFrame(sorted(
    (load_complex(fp) for fp in PER_COMPLEX.glob("*.csv")),
    key=lambda r: r["complex_id"],
))

# pocket engagement / good-pose bookkeeping from redocking Stage 8
gpr = pd.read_csv(COMPARISON / "good_pose_representative.csv")[
    ["complex_id", "n_active_residues_contacted", "n_good_poses"]
]
complex_df = complex_df.merge(gpr, on="complex_id", how="left")
complex_df["good_pose"] = complex_df["n_active_residues_contacted"] >= 3  # Stage-8 threshold

print(f"{len(complex_df)} complexes, {complex_df.protein.nunique()} proteins")
print(complex_df.role.value_counts().to_dict())
complex_df.head()

### 1a. Filter out the off-target ("bad") poses up front

The scored pose per complex is HADDOCK3's own top-ranked model. In some
complexes that model never actually enters the CDD pocket — it sits on the
membrane face / off-target — and `redocking/RESULTS.md` §3 shows those
off-target poses can carry *very* favourable `haddock_score` (burial) while
meaning nothing about GA binding. Earlier runs of this notebook found the
`haddock_score` bimodality was largely this on-target/off-target split, not
biology.

**So we drop them here, once, before anything else runs.** "Bad pose" =
`n_active_residues_contacted < 3` (same threshold as `good_pose`; the Stage-8
GMM(2) on contact count has component means 0.5 vs 11, so 3 is deep in the
valley). Everything downstream — `protein_df`, the GMMs, the disagreement
lists, the 2-D fit, the clade and supervised sections — then operates on
on-target poses only.

In [ ]:
complex_df_all = complex_df.copy()                      # keep the unfiltered table for reference
bad = complex_df_all[~complex_df_all["good_pose"]]

print(f"dropping {len(bad)} off-target complexes "
      f"({len(bad)}/{len(complex_df_all)} = {len(bad)/len(complex_df_all):.0%})")
print("  by role:", bad["role"].value_counts().to_dict())
print("  proteins losing >=1 macro-conformation:",
      bad.groupby("protein").size().to_dict())

complex_df = complex_df_all[complex_df_all["good_pose"]].reset_index(drop=True)

lost_all = sorted(set(complex_df_all["protein"]) - set(complex_df["protein"]))
print(f"\nkept {len(complex_df)} complexes, {complex_df.protein.nunique()} proteins "
      f"({complex_df_all.protein.nunique() - complex_df.protein.nunique()} proteins dropped entirely: {lost_all or 'none'})")
per_prot = complex_df.groupby("protein").size()
print(f"macro-conformations kept per protein: {per_prot.value_counts().sort_index().to_dict()}")

## 2. Attach the biological labels (and a second, sequence-only opinion)

- **`bio_label`** (0/1) — the **strict high-confidence split**:
  `NPF_LDA_kernel/config/config.yaml`'s `hc_importers` (12) and
  `hc_non_importers` (21), the proteins with the strongest experimental
  evidence. Everything outside those two lists is `NaN` and dropped from every
  label-agreement statistic. In the redocking corpus this leaves **29 of 48
  proteins** (10 importer / 19 non-importer). This is the label the hypothesis
  says may be contaminated.
- **`assay_label`** — the label as originally shipped in
  `NPF_LDA_kernel/results/ga_classifier/labels.tsv` (Chiba et al. 2015;
  Jørgensen et al. 2017), carried only as a cross-reference column. It differs
  from the strict split — e.g. **NPF2.7** is in `hc_importers` but
  `labels.tsv = 0` — so disagreements between `bio_label` and `assay_label`
  are themselves worth a look.
- **`bio_broad`** — the wider `ga_importers` list from `NPF_LDA_kernel`'s
  config (20 positives; adds lower-confidence importers).
- **`lda_score`** — continuous decision score of `NPF_LDA_kernel`'s
  `track_b_spectrum_k2` sequence classifier. On this strict HC split the
  sequence classifiers are essentially perfect (LOO AUC 0.996 k=2 / 1.00 k=3,
  `hc/summary.tsv`). A second opinion that is **not** the assay label: when the
  energy *and* this score both disagree with `bio_label`, that's the strongest
  contamination candidate.
- **`clade`** — NPF subfamily (`NPF1`..`NPF8`) parsed from the name.

In [ ]:
import json

# --- primary label: strict high-confidence split -------------------------
#     NPF_LDA_kernel/config/config.yaml  hc_importers / hc_non_importers
HC_IMPORTERS = {
    "NPF3.1", "NPF4.1", "NPF2.12", "NPF2.13", "NPF2.10", "NPF2.5",
    "NPF2.7", "NPF2.3", "NPF2.4", "NPF4.2", "NPF1.1", "NPF1.2",
}
HC_NON_IMPORTERS = {
    "NPF8.1", "NPF8.2", "NPF8.3", "NPF8.4", "NPF8.5", "NPF6.1", "NPF6.2",
    "NPF6.3", "NPF6.4", "NPF7.1", "NPF7.2", "NPF7.3", "NPF5.8", "NPF5.9",
    "NPF5.10", "NPF5.11", "NPF5.12", "NPF5.13", "NPF5.14", "NPF5.15", "NPF5.16",
}
hc_label_by_npf = {**{p: 1 for p in HC_IMPORTERS}, **{p: 0 for p in HC_NON_IMPORTERS}}

# --- original assay label (labels.tsv), reference column only ------------
_assay = pd.read_csv(LDA_KERNEL / "labels.tsv", sep="\t", header=None, names=["protein", "v"])
assay_by_npf = dict(zip(_assay["protein"].str.rsplit("_", n=1).str[0], _assay["v"]))

# --- wider positive list (NPF_LDA_kernel config.yaml `ga_importers`) -----
BIO_BROAD = {
    "NPF1.1", "NPF1.2", "NPF2.1", "NPF2.3", "NPF2.4", "NPF2.5", "NPF2.6",
    "NPF2.7", "NPF2.10", "NPF2.11", "NPF2.12", "NPF2.13", "NPF2.14",
    "NPF3.1", "NPF4.1", "NPF4.2", "NPF5.1", "NPF5.2", "NPF5.6", "NPF5.7",
}

# --- sequence classifier decision score --------------------------------
lda_scores = json.load(open(LDA_KERNEL / "track_b_spectrum_k2.json"))["scores"]
lda_by_npf = {k.rsplit("_", 1)[0]: v for k, v in lda_scores.items()}


def npf_of(protein):     # 'NPF3.1_Q9SX20' -> 'NPF3.1'
    return protein.rsplit("_", 1)[0]


def clade_of(protein):   # 'NPF3.1_Q9SX20' -> 'NPF3'
    return npf_of(protein).split(".")[0]


for df in (complex_df,):
    df["npf"] = df["protein"].map(npf_of)
    df["clade"] = df["protein"].map(clade_of)
    df["bio_label"] = df["npf"].map(hc_label_by_npf)       # 1 / 0 / NaN  (strict HC split)
    df["assay_label"] = df["npf"].map(assay_by_npf)        # original labels.tsv, reference only
    df["bio_broad"] = df["npf"].isin(BIO_BROAD).astype(int)
    df["lda_score"] = df["npf"].map(lda_by_npf)

n_lab = complex_df["bio_label"].notna().sum()
print(f"complexes with a HC label: {n_lab}/{len(complex_df)}  "
      f"(positive={int(complex_df['bio_label'].sum())})")
print(f"proteins with a HC label: {complex_df.loc[complex_df.bio_label.notna(),'protein'].nunique()}"
      f"/{complex_df.protein.nunique()}")
missing = sorted(complex_df.loc[complex_df.bio_label.isna(), "npf"].unique())
print(f"redocked but outside the HC split ({len(missing)}): {missing}")
# where the strict split and the original assay label disagree:
disc = complex_df.dropna(subset=["bio_label", "assay_label"])
disc = disc[disc["bio_label"] != disc["assay_label"]]["npf"].unique()
print(f"HC vs labels.tsv disagreements: {sorted(disc)}")

### Collapse to one row per protein

Each protein was redocked in 3 macro-conformations. For the protein-level view
we keep its **best** (most binder-like) pose per metric — a protein only needs
*one* competent conformation to be a real transporter, and `redocking/RESULTS.md`
§3 shows non-importers that fail in one conformation usually succeed in another.

In [ ]:
def best_per_protein(df, metric):
    # most binder-like value across the protein's 3 macro-conformations;
    # lower = better for every metric in METRICS, so take the min.
    idx = df.groupby("protein")[metric].idxmin()
    return df.loc[idx].set_index("protein")[metric]


meta_cols = ["protein", "npf", "clade", "form", "bio_label", "assay_label", "bio_broad", "lda_score", "role"]
protein_df = complex_df.drop_duplicates("protein")[meta_cols].set_index("protein").copy()
for m in METRICS:
    protein_df[m] = best_per_protein(complex_df, m)
protein_df["n_good_poses_max"] = complex_df.groupby("protein")["n_good_poses"].max()
# protein-level pose-quality flag: does the protein have >=1 competent conformation?
protein_df["good_pose"] = complex_df.groupby("protein")["good_pose"].any()
protein_df = protein_df.reset_index()

print(f"{len(protein_df)} proteins; with HC label: {protein_df.bio_label.notna().sum()} "
      f"(pos={int(protein_df.bio_label.sum())})")
protein_df.head()

## 3. Univariate distributions — is anything visibly bimodal?

Histogram + rug of each energy metric, coloured by biological label. A real
binder/non-binder axis should look like **two humps**; if it looks like one
skewed hump, no amount of GMM machinery will rescue it.

In [ ]:
def dist_plot(df, metric, level):
    desc = METRICS[metric]
    d = df.copy()
    d["label"] = d["bio_label"].map({1: "importer", 0: "non-importer"}).fillna("unlabelled")
    fig = px.histogram(
        d, x=metric, color="label", marginal="rug", nbins=40, opacity=0.65,
        color_discrete_map={"importer": "#2ca02c", "non-importer": "#7f7f7f",
                            "unlabelled": "#cccccc"},
        title=f"[{level}] {metric}  —  {desc}",
    )
    fig.update_layout(barmode="overlay", height=380, bargap=0.02)
    fig.show()

    # cheap shape diagnostics
    x = df[metric].dropna().values
    k2, dip_p = stats.normaltest(x)
    print(f"  {metric:26s} n={len(x):3d}  skew={stats.skew(x):+.2f}  "
          f"kurtosis={stats.kurtosis(x):+.2f}  D'Agostino normality p={dip_p:.1e}")


for m in METRICS:
    dist_plot(complex_df, m, "per-complex")
for m in METRICS:
    dist_plot(protein_df, m, "per-protein (best pose)")

## 4. Unsupervised 1-D GMM per metric

For each metric: standardize, fit `GaussianMixture` with `k = 1, 2, 3`
(`n_init=10`), pick `k` by **BIC**. If `k ≥ 2`, take the 2-component fit,
label the **favourable** component (lower energy for HADDOCK / Rosetta totals,
also lower for `fa_rep`), then score that binary call against:

- `bio_label` — Adjusted Rand Index, balanced accuracy, confusion matrix;
- `good_pose` — now `n/a` (off-target poses removed in §1a); the line stays as
  a guard that the filter held;
- and separately, the **ROC AUC of the raw metric** vs `bio_label` (a
  threshold-free supervised ceiling that doesn't depend on the GMM at all).

In [ ]:
def fit_1d_gmm(x, ks=(1, 2, 3), n_init=10, seed=RNG):
    xs = StandardScaler().fit_transform(x.reshape(-1, 1))
    out = {}
    for k in ks:
        g = GaussianMixture(k, n_init=n_init, random_state=seed).fit(xs)
        out[k] = {"bic": g.bic(xs), "aic": g.aic(xs), "model": g, "xs": xs}
    best_k = min(ks, key=lambda k: out[k]["bic"])
    return out, best_k


def favourable_component(g):
    # lower component mean = more binder-like for every metric here
    return int(np.argmin(g.means_.ravel()))


def eval_metric(df, metric, level):
    desc = METRICS[metric]
    sub = df.dropna(subset=[metric]).copy()
    x = sub[metric].values
    out, best_k = fit_1d_gmm(x)
    bic = {k: round(out[k]["bic"], 1) for k in out}
    g2 = out[2]["model"]
    xs2 = out[2]["xs"]
    fav = favourable_component(g2)
    comp = g2.predict(xs2)
    sub["gmm_binder"] = (comp == fav).astype(int)

    # component summary in original units
    means = StandardScaler().fit(x.reshape(-1, 1)).inverse_transform(g2.means_).ravel()
    order = np.argsort(means)
    comp_tbl = pd.DataFrame({
        "component": [f"C{i}" + ("  <-favourable" if i == fav else "") for i in order],
        "mean": means[order].round(1),
        "weight": g2.weights_[order].round(2),
    })

    labelled = sub.dropna(subset=["bio_label"])
    ari = adjusted_rand_score(labelled["bio_label"], labelled["gmm_binder"]) if len(labelled) else np.nan
    bacc = balanced_accuracy_score(labelled["bio_label"], labelled["gmm_binder"]) if len(labelled) else np.nan
    try:
        auc = roc_auc_score(labelled["bio_label"], -labelled[metric])  # lower metric = binder
    except ValueError:
        auc = np.nan
    conf = pd.crosstab(labelled["bio_label"].map({1: "importer", 0: "non-importer"}),
                       labelled["gmm_binder"].map({1: "gmm:binder", 0: "gmm:non-binder"}))
    # is the split just pose quality? (off-target poses were filtered in 1a, so
    # this should now be uninformative -- kept as a check that the filter held)
    if sub["good_pose"].nunique() > 1:
        pose_ari = adjusted_rand_score(sub["good_pose"].astype(int), sub["gmm_binder"])
        pose_note = (f"vs good_pose:  ARI={pose_ari:+.3f}   "
                     f"<- high => the bimodality is POSE QUALITY, not biology")
    else:
        pose_ari = np.nan
        pose_note = "vs good_pose:  n/a — all off-target poses filtered in 1a"

    print(f"\n=== [{level}] {metric} — {desc}")
    print(f"BIC(k=1/2/3) = {bic}   -> BIC picks k={best_k}")
    print(comp_tbl.to_string(index=False))
    print(f"\nvs HC label (n={len(labelled)}):  ARI={ari:+.3f}   balanced_acc={bacc:.3f}   "
          f"raw-metric ROC AUC={auc:.3f}")
    print(conf.to_string())
    print("\n" + pose_note)
    return {"level": level, "metric": metric, "bic_best_k": best_k,
            "bic1": bic[1], "bic2": bic[2], "bic3": bic[3],
            "fav_mean": comp_tbl.loc[comp_tbl.component.str.contains("favourable"), "mean"].iloc[0],
            "fav_weight": comp_tbl.loc[comp_tbl.component.str.contains("favourable"), "weight"].iloc[0],
            "ARI_label": ari, "bal_acc": bacc, "raw_AUC": auc, "ARI_goodpose": pose_ari}


rows = []
for m in METRICS:
    rows.append(eval_metric(complex_df, m, "per-complex"))
for m in METRICS:
    rows.append(eval_metric(protein_df, m, "per-protein"))
gmm1d_summary = pd.DataFrame(rows)

In [ ]:
# headline table
gmm1d_summary.style.format({
    "bic1": "{:.0f}", "bic2": "{:.0f}", "bic3": "{:.0f}",
    "fav_mean": "{:.1f}", "fav_weight": "{:.2f}",
    "ARI_label": "{:+.3f}", "bal_acc": "{:.3f}", "raw_AUC": "{:.3f}", "ARI_goodpose": "{:+.3f}",
}).set_caption("1-D GMM per metric: does BIC want 2 components, and does the split track biology or just pose quality?")

## 5. The disagreement list — where the energy disputes the assay

Take the **best-separating metric** (largest `raw_AUC` at protein level that is
*not* explained by pose quality), threshold proteins by the GMM's favourable
component, and print the two disagreement classes:

- **HC non-importer, energy says binder** → candidate contamination / real
  GA transporters currently mislabelled negative. Cross-referenced with
  `lda_score` (sequence classifier) and `assay_label`: if either *also* leans
  positive, it's a stronger candidate.
- **HC importer, energy says non-binder** → candidate weak / slow importers
  — consistent with the "same clade, very different rate" point.

In [ ]:
def disagreement_report(metric):
    desc = METRICS[metric]
    sub = protein_df.dropna(subset=[metric, "bio_label"]).copy()
    x = sub[metric].values
    out, _ = fit_1d_gmm(x)
    g2 = out[2]["model"]
    fav = favourable_component(g2)
    sub["gmm_binder"] = (g2.predict(out[2]["xs"]) == fav).astype(int)
    # decision boundary in original units (midpoint between component means, informational)
    means = StandardScaler().fit(x.reshape(-1, 1)).inverse_transform(g2.means_).ravel()
    boundary = means.mean()

    cols = ["protein", "clade", "form", metric, "lda_score", "assay_label", "n_good_poses_max", "role"]
    print(f"metric: {metric}   ({desc})")
    print(f"GMM favourable-component mean vs other: {sorted(means.round(1))}   "
          f"~midpoint {boundary:.1f}\n")

    fp = sub[(sub.bio_label == 0) & (sub.gmm_binder == 1)].sort_values(metric)
    print(f"--- HC NON-importer, energy says BINDER  (n={len(fp)}) "
          f"— candidate contamination ---")
    show = fp[cols].copy()
    show["seq_clf_also_positive"] = show["lda_score"] > 0
    print(show.to_string(index=False))

    fn = sub[(sub.bio_label == 1) & (sub.gmm_binder == 0)].sort_values(metric, ascending=False)
    print(f"\n--- HC importer, energy says NON-binder  (n={len(fn)}) "
          f"— candidate weak/slow importers ---")
    show = fn[cols].copy()
    show["seq_clf_also_negative"] = show["lda_score"] < 0
    print(show.to_string(index=False))
    return fp, fn


# pick the metric with the best protein-level raw_AUC (pose quality no longer a
# confounder here -- off-target poses were filtered in 1a).
cand = gmm1d_summary.query("level == 'per-protein'").copy()
cand = cand[cand["ARI_goodpose"].isna() | (cand["ARI_goodpose"] < 0.25)]
best_metric = cand.sort_values("raw_AUC", ascending=False)["metric"].iloc[0]
print(f"best protein-level separating metric: {best_metric}\n")
fp_main, fn_main = disagreement_report(best_metric)

print("\n\n================  same report for every metric  ================")
for m in METRICS:
    print("\n" + "=" * 70)
    disagreement_report(m)

## 6. 2-D GMM on (HADDOCK × Rosetta)

Does combining the two force fields give a cleaner mixture than either alone?
Fit on standardized `[haddock_score, total_score_haddock_pose]` (and a variant
swapping in `lig_twobody_sum`), `k = 1..3`, BIC-selected. Scatter: colour =
GMM component, marker symbol = biological label.

In [ ]:
def gmm_2d(df, feats, level, ks=(1, 2, 3)):
    sub = df.dropna(subset=list(feats) + ["bio_label"]).copy()
    X = StandardScaler().fit_transform(sub[list(feats)].values)
    fits = {}
    for k in ks:
        g = GaussianMixture(k, n_init=10, random_state=RNG).fit(X)
        fits[k] = (g.bic(X), g)
    best_k = min(ks, key=lambda k: fits[k][0])
    g = fits[best_k][1]
    sub["comp"] = g.predict(X)

    # orient: favourable component = lower mean haddock_score
    hadd_means = [sub.loc[sub.comp == c, "haddock_score"].mean() for c in range(best_k)]
    fav = int(np.nanargmin(hadd_means))
    sub["gmm_binder"] = (sub["comp"] == fav).astype(int)
    ari = adjusted_rand_score(sub["bio_label"], sub["gmm_binder"])

    bic_str = {k: round(fits[k][0], 1) for k in ks}
    print(f"[{level}] feats={feats}  BIC={bic_str}  -> k={best_k}   ARI vs label={ari:+.3f}")
    print(pd.crosstab(sub["bio_label"].map({1: 'importer', 0: 'non-importer'}),
                      sub["gmm_binder"].map({1: 'gmm:binder', 0: 'gmm:non-binder'})).to_string())

    d = sub.copy()
    d["label"] = d["bio_label"].map({1: "importer", 0: "non-importer"})
    d["component"] = "C" + d["comp"].astype(str)
    fig = px.scatter(
        d, x=feats[0], y=feats[1], color="component", symbol="label",
        hover_name="protein", title=f"[{level}] 2-D GMM  {feats}  (k={best_k})",
        symbol_map={"importer": "circle", "non-importer": "x"},
    )
    fig.update_traces(marker_size=11, marker_line_width=1)
    fig.update_layout(height=520)
    fig.show()
    return sub


_ = gmm_2d(protein_df, ("haddock_score", "total_score_haddock_pose"), "per-protein")
_ = gmm_2d(protein_df, ("haddock_score", "lig_twobody_sum"), "per-protein")
_ = gmm_2d(complex_df, ("haddock_score", "total_score_haddock_pose"), "per-complex")

## 7. Clade structure — the "same clade, very different rate" confounder

The PI's point: GA-transport rate is graded, and NPFs in one clade span that
range. If most of the energy spread is *between clades* rather than
*between importer/non-importer within a clade*, then a global bimodal model is
the wrong shape and a per-clade or continuous model is more honest.

In [ ]:
def clade_plot(metric):
    desc = METRICS[metric]
    d = protein_df.dropna(subset=[metric]).copy()
    d["label"] = d["bio_label"].map({1: "importer", 0: "non-importer"}).fillna("unlabelled")
    fig = px.strip(d.sort_values("clade"), x="clade", y=metric, color="label",
                   color_discrete_map={"importer": "#2ca02c", "non-importer": "#7f7f7f",
                                       "unlabelled": "#cccccc"},
                   title=f"{metric} by clade  —  {desc}", stripmode="overlay")
    fig.update_traces(marker_size=10, jitter=0.3)
    fig.update_layout(height=420)
    fig.show()


for m in ["haddock_score", "total_score_haddock_pose", "lig_twobody_sum"]:
    clade_plot(m)

# per-clade importer/non-importer counts and mean metric
tbl = (protein_df.assign(lbl=protein_df.bio_label.map({1: "imp", 0: "non"}))
       .groupby(["clade", "lbl"])
       .agg(n=("protein", "size"),
            haddock=("haddock_score", "mean"),
            rosetta_total=("total_score_haddock_pose", "mean"),
            lig_twobody=("lig_twobody_sum", "mean"))
       .round(1))
print(tbl.to_string())

# variance decomposition: how much of each metric is 'between clade' vs residual
for m in ["haddock_score", "total_score_haddock_pose", "lig_twobody_sum"]:
    d = protein_df.dropna(subset=[m])
    grand = d[m].mean()
    ss_tot = ((d[m] - grand) ** 2).sum()
    ss_between = d.groupby("clade")[m].apply(lambda s: len(s) * (s.mean() - grand) ** 2).sum()
    print(f"{m:26s}  between-clade variance share = {ss_between / ss_tot:.2f}")

## 8. Supervised upper bound & verdict

Best separation each energy (and the multi-metric combos) can achieve against
`bio_label` (the strict HC split) at the protein level, next to the
sequence-only LDA kernel — which on this split is essentially perfect
(`hc/summary.tsv` LOO AUC 0.996 k=2 / 1.00 k=3). This is the ceiling: if even a
*supervised* fit can't separate the classes, the *unsupervised* GMM never will.

Two AUCs are shown per feature set:

- **rank AUC** — threshold-free Mann–Whitney AUC of the raw feature (for
  1-D feature sets), sign-oriented so "lower energy ⇒ importer". Depends on
  nothing but the data.
- **CV AUC** — 5-fold *stratified* logistic regression
  (`cross_val_predict`). Leave-One-Out is deliberately **not** used here: with
  a ~10/29 class imbalance and a non-informative feature, LOO's intercept shift
  manufactures a spurious anti-correlation between the held-out prediction and
  its label, driving AUC toward 0 for reasons that have nothing to do with the
  feature. 5-fold stratified CV does not have that pathology. (With only ~10
  importers the 5-fold CV AUC is itself noisy — read it together with the
  threshold-free rank AUC, not on its own.)

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RNG)


def cv_auc(df, feats):
    sub = df.dropna(subset=list(feats) + ["bio_label"]).copy()
    X = sub[list(feats)].values
    y = sub["bio_label"].values.astype(int)
    pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    p = cross_val_predict(pipe, X, y, cv=CV, method="predict_proba")[:, 1]
    cv = roc_auc_score(y, p)
    rank = roc_auc_score(y, -sub[feats[0]]) if len(feats) == 1 else np.nan
    return rank, cv, len(y)


print("protein-level vs HC label     (rank AUC = raw feature | CV AUC = 5-fold logistic)")
for feats in [("haddock_score",), ("total_score_haddock_pose",), ("lig_twobody_sum",),
              ("lig_twobody_min",), ("fa_rep_haddock_pose",),
              ("haddock_score", "total_score_haddock_pose"),
              ("haddock_score", "total_score_haddock_pose", "lig_twobody_sum")]:
    rank, cv, n = cv_auc(protein_df, feats)
    rank_s = f"{rank:.3f}" if np.isfinite(rank) else "  -  "
    print(f"  rank={rank_s}   CV={cv:.3f}   (n={n})   {feats}")

seq = protein_df.dropna(subset=["lda_score", "bio_label"])
print(f"\n  rank={roc_auc_score(seq.bio_label, seq.lda_score):.3f}   CV=  -     (n={len(seq)})   "
      f"(lda_score,)  <- spectrum-k2 LOO scores (fit on 45); dedicated HC-set "
      f"LOO AUC 0.996 (k2) / 1.00 (k3), hc/summary.tsv")

# apo/holo confound check: 'holo' complexes are the real GA1 co-folds and are
# almost all importers, so any metric that tracks apo-vs-holo will look
# predictive of the label without being about binding at all.
print("\nlabel x redocking form (holo = built on a real ABCfold GA1 co-fold pose):")
print(pd.crosstab(protein_df.bio_label.map({1: "importer", 0: "non-importer"}),
                  protein_df.form).to_string())

### Reading of the result — decision guide

| observation | reading |
|---|---|
| No metric has `BIC` preferring `k ≥ 2` | energies are unimodal — **no bimodal binder/non-binder model exists**; hypothesis not testable this way. |
| `k ≥ 2` but the minority component is a handful of proteins, or `ARI_label` ≈ 0 | the two humps are outliers / weak-engagement poses, not the biology. |
| `k ≥ 2`, but `ARI_label` ≈ 0 and rank/CV AUC ≈ 0.5 | there may be structure in the energy, but it is **orthogonal to the GA-importer label** — clade, graded activity, or noise. |
| rank AUC ≳ 0.7, short and chemically sensible disagreement list, **not** explained by apo/holo form | **supports the PI hypothesis** — take the disagreement list forward. |
| between-clade variance share ≳ 0.5 | a global mixture is the wrong shape; go per-clade or continuous. |

### Observed outcome — strict HC split, **off-target poses filtered (§1a)** (re-run 2026-08-28)

**Dropping the 13 off-target complexes up front (all `non_importer`; 143 → 130
complexes, 0 proteins lost, 78 complexes / 29 proteins carry a HC label) does
not produce a binder / non-binder model. One metric moved, none separated.**

- **`lig_twobody_sum` tightened but did not become bimodal.** Its protein-level
  rank AUC vs the HC label rose from **0.64 → 0.72** (and the
  `haddock + total + lig_twobody_sum` logistic CV AUC 0.63 → 0.68) — the only
  real effect of the filter. But BIC still prefers **k = 1** for it (favourable
  component mean −1.8, weight 0.73), and its `ARI_label` is −0.07. A modestly
  informative *direction*, not a separable *cluster*.
- **The `haddock_score` bimodality survives the filter and is still not
  biological.** BIC still picks **k = 3** (per-complex and per-protein), with a
  ~8–25 % minority component at much less favourable scores (−28 to −52 vs
  −505 to −524). Removing the `< 3`-contact poses did not collapse it — the
  second mode is now weak-engagement / poor-score poses that still clear the
  3-contact bar, and it still has `ARI_label ≈ −0.02…−0.07`.
- **Still no separation, supervised or not.** Protein-level rank AUC vs the HC
  label: `haddock_score` 0.54, `total_score` 0.45, `lig_twobody_sum` **0.72**,
  `lig_twobody_min` 0.54, `fa_rep` 0.33. Every 5-fold logistic CV AUC is
  0.4–0.68. The sequence score's rank AUC on the same proteins is **0.91**.
  `fa_rep` still only looks predictive because of the apo/holo confound
  (**all 5 `holo` poses are HC importers, 0 non-importers are `holo`**).
- **Clade confound unchanged** (filtering removed no clade): HC importers are
  NPF1×2 / NPF2×7 / NPF3×1, all 19 HC non-importers are NPF5–NPF8; between-clade
  variance share 0.08 / 0.35 / 0.33.
- **Disagreement list, `lig_twobody_sum`, per protein:** after filtering **all
  10 HC importers** land in the favourable component (0 "energy says
  non-binder") — but so do **16 of 19** HC non-importers, so it still doesn't
  discriminate. Of those 16, only NPF7.2, NPF5.10, NPF5.15 also have a positive
  sequence score; NPF7.2 is the one name that recurs across label sets and
  metrics. NPF2.7 (HC importer, `assay_label = 0`) sits on the binder side and
  is again unresolved by the energies.

**Bottom line:** removing the bad poses cleaned up `lig_twobody_sum` (rank AUC
0.64 → 0.72) but did **not** create a bimodal, biology-aligned split, and no
energy metric tells HC importers from HC non-importers. The `haddock_score`
bimodality was never *only* membrane junk — a weak-engagement second mode
persists after filtering — and it still carries no importer signal. The
limiting factors remain the restraint-driven docking and the near-total
clade/label confound. The only lead that survives — NPF7.2 — comes from the
**sequence** score.

### Why this is an expected negative, and what would actually move it

1. **HADDOCK's AIRs are built from the same 35 CDD residues for every protein**
   (`redocking/src/define_active_passive.py`), so the docking is *steered* into
   the same pocket regardless of true importer status — `redocking/RESULTS.md`
   §4/§7 flag exactly this. A **blind / unrestrained** re-dock of a candidate
   subset is the single most informative next experiment.
2. **One un-relaxed pose per complex, two force fields that already disagree on
   sign** (`RESULTS.md` §4). Fold in the `mmgbsa/` per-residue GB decomposition
   once it has run — a third, restraint-free energy — before concluding
   anything about individual proteins.
3. **If GA transport is graded**, a binary label is the wrong target. Ask the
   PI for any quantitative rate/`Km` numbers and redo §4–8 as a regression; a
   continuous model may find signal a 2-class GMM cannot.
4. The one method that *does* separate the HC classes is **sequence, not
   structure** (LDA kernel, LOO AUC ~1.0 on this split — though on n=33 with a
   near-perfect clade/label confound that is itself partly a phylogeny signal;
   see `pocket_chemistry_embedding.ipynb`). The most efficient contamination
   screen is proteins where the **sequence classifier and the assay disagree**,
   redocked *blind* — not the restraint-driven energy used here.